# BÀI THỰC HÀNH 08. XỬ LÝ ẢNH MÀU

Bài thực hành mở rộng các kỹ thuật xử lý ảnh xám sang ảnh màu.

| Phần | Nội dung |
|------|----------|
| 1 | Không gian màu RGB / HSV / Lab |
| 2 | Histogram ảnh màu (RGB + Hue) |
| 3 | Cân bằng histogram (V, L\*, CLAHE) |
| 4 | Phân ngưỡng tự động (Otsu, multi-level, adaptive) |
| 5 | Phát hiện biên màu kiểu Canny |
| 6 | Phân đoạn vùng (region growing, K-means, watershed, split-merge) |
| 7 | Phân đoạn biên (edge-to-region, active contour) |
| 8 | Bài tập tổng hợp |

## 0. Các hàm dùng chung

Đọc và hiển thị ảnh màu (RGB).

In [ ]:
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# ---- I/O ----
def load_image(path):
    img = cv2.imread(path)
    if img is None:
        print(f'Cannot read: {path}')
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def simulate_night(img_rgb, factor=0.25):
    '''Giam do sang de mo phong canh ban dem.'''
    return np.clip(img_rgb.astype(np.float32) * factor, 0, 255).astype(np.uint8)

# ---- Hien thi ----
def show(img, title='', figsize=(5, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap='gray' if img.ndim == 2 else None)
    plt.title(title); plt.axis('off'); plt.tight_layout(); plt.show()

def show_images(images, titles=None, cols=3, figsize=(15, 5)):
    n = len(images)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i], cmap='gray' if images[i].ndim == 2 else None)
            if titles and i < len(titles):
                ax.set_title(titles[i])
            ax.axis('off')
        else:
            ax.set_visible(False)
    plt.tight_layout(); plt.show()

# ---- Overlay ----
def overlay_mask(img_rgb, mask, color=(255, 0, 0), alpha=0.4):
    out = img_rgb.copy().astype(np.float32)
    m = mask > 0
    out[m] = (1-alpha)*out[m] + alpha*np.array(color, np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)

def colorize_labels(label_map):
    k = int(label_map.max()) + 1
    rng = np.random.default_rng(0)
    colors = rng.integers(40, 220, size=(k, 3), dtype=np.uint8)
    return colors[label_map]

print('Utilities loaded.')

## 1. Không gian màu và chuyển đổi

### 1.1 Lý thuyết

Ảnh màu là hàm vector: $\mathbf{I}(x,y)=[R,G,B]^T$.

HSV tách biệt **Hue** (loại màu), **Saturation** (độ tinh khiết), **Value** (độ sáng).

Công thức chuyển đổi (với RGB chuẩn hóa $[0,1]$):
$$V=C_{max},\quad S=\frac{\Delta}{C_{max}},\quad
H=\begin{cases}0 & \Delta=0\\ 60\tfrac{G-B}{\Delta} & C_{max}=R\\ 60\bigl(\tfrac{B-R}{\Delta}+2\bigr) & C_{max}=G\\ 60\bigl(\tfrac{R-G}{\Delta}+4\bigr) & C_{max}=B\end{cases}$$

### 1.2 Triển khai bằng NumPy (§2.3.1)

In [ ]:
def rgb_to_hsv_numpy(img):
    '''RGB uint8 -> HSV float32: H in [0,360], S,V in [0,1].'''
    f = img.astype(np.float32) / 255.0
    R, G, B = f[:,:,0], f[:,:,1], f[:,:,2]
    Cmax = np.maximum.reduce([R, G, B])
    Cmin = np.minimum.reduce([R, G, B])
    delta = Cmax - Cmin

    H = np.zeros_like(Cmax)
    mask = delta != 0
    idx = (Cmax == R) & mask; H[idx] = ((G[idx]-B[idx])/delta[idx]) % 6
    idx = (Cmax == G) & mask; H[idx] = (B[idx]-R[idx])/delta[idx] + 2
    idx = (Cmax == B) & mask; H[idx] = (R[idx]-G[idx])/delta[idx] + 4
    H = H * 60

    S = np.zeros_like(Cmax)
    S[Cmax != 0] = delta[Cmax != 0] / Cmax[Cmax != 0]
    return np.stack([H, S, Cmax], axis=-1)


def hsv_to_rgb_numpy(hsv):
    '''HSV float32 (H in [0,360], S,V in [0,1]) -> RGB uint8.'''
    H, S, V = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    C = V * S
    X = C * (1 - np.abs((H/60) % 2 - 1))
    m = V - C
    R = np.zeros_like(H); G = np.zeros_like(H); B = np.zeros_like(H)
    for lo, hi, r, g, b in [
        (  0,  60, C, X, 0), ( 60, 120, X, C, 0), (120, 180, 0, C, X),
        (180, 240, 0, X, C), (240, 300, X, 0, C), (300, 360, C, 0, X)
    ]:
        c = (lo <= H) & (H < hi)
        R[c], G[c], B[c] = r if np.isscalar(r) else r[c], \
                           g if np.isscalar(g) else g[c], \
                           b if np.isscalar(b) else b[c]
    return (np.stack([(R+m),(G+m),(B+m)], axis=-1)*255).astype(np.uint8)

In [ ]:
img_leaves = load_image('leaves.png')
hsv_np  = rgb_to_hsv_numpy(img_leaves)
rgb_back = hsv_to_rgb_numpy(hsv_np)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
rows = [
    [img_leaves[:,:,0], img_leaves[:,:,1], img_leaves[:,:,2], img_leaves],
    [hsv_np[:,:,0],     hsv_np[:,:,1],     hsv_np[:,:,2],     rgb_back],
]
titles = ['R','G','B','RGB goc', 'H (NumPy)','S (NumPy)','V (NumPy)','RGB phuc hoi']
cmaps  = ['Reds','Greens','Blues',None,'hsv','gray','gray',None]
for row_i, row_data in enumerate(rows):
    for j, (d, t, cm) in enumerate(zip(row_data, titles[row_i*4:], cmaps[row_i*4:])):
        ax = axes[row_i, j]
        ax.imshow(d) if cm is None else ax.imshow(d, cmap=cm)
        ax.set_title(t); ax.axis('off')
plt.suptitle('Cac kenh RGB va HSV (NumPy implementation)', fontsize=13)
plt.tight_layout(); plt.show()

err = np.mean(np.abs(img_leaves.astype(np.float32) - rgb_back.astype(np.float32)))
print(f'Sai so phuc hoi RGB: {err:.4f} (can < 1.0)')

# So sanh H channel voi OpenCV
hsv_cv = cv2.cvtColor(img_leaves, cv2.COLOR_RGB2HSV)
diff_H = np.mean(np.abs(hsv_np[:,:,0]/2.0 - hsv_cv[:,:,0].astype(np.float32)))
print(f'Sai lech H so voi OpenCV: {diff_H:.4f}')

## 2. Histogram cho ảnh màu

### 2.1 Triển khai NumPy (§3.2)

In [ ]:
def color_histogram_numpy(img):
    '''Histogram RGB bang vong lap — triển khai giao duc (§3.2).'''
    hist_r = np.zeros(256); hist_g = np.zeros(256); hist_b = np.zeros(256)
    for pixel in img.reshape(-1, 3):
        r, g, b = pixel
        hist_r[r] += 1; hist_g[g] += 1; hist_b[b] += 1
    return hist_r, hist_g, hist_b

def color_histogram_fast(img):
    '''Histogram RGB nhanh bang np.bincount.'''
    return (np.bincount(img[:,:,c].ravel(), minlength=256).astype(np.float64)
            for c in range(3))

def plot_rgb_hist(img, title='RGB Histogram', ax=None):
    hr = np.bincount(img[:,:,0].ravel(), minlength=256).astype(np.float64)
    hg = np.bincount(img[:,:,1].ravel(), minlength=256).astype(np.float64)
    hb = np.bincount(img[:,:,2].ravel(), minlength=256).astype(np.float64)
    x = np.arange(256)
    standalone = ax is None
    if standalone:
        _, ax = plt.subplots(figsize=(9, 3))
    ax.plot(x, hr, 'r', alpha=0.8, lw=1.5, label='R')
    ax.plot(x, hg, 'g', alpha=0.8, lw=1.5, label='G')
    ax.plot(x, hb, 'b', alpha=0.8, lw=1.5, label='B')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
    if standalone:
        plt.tight_layout(); plt.show()

### 2.2 Bài tập 3.3.1 — Histogram RGB cho 3 ảnh (ngày / đêm / ngoài trời)

Vì bộ ảnh không có ảnh đêm thực, ta **mô phỏng ảnh đêm** bằng cách giảm độ sáng `leaves.png`.

In [ ]:
img_day   = load_image('leaves.png')          # Ban ngay — la mua thu nhieu mau
img_night = simulate_night(img_day, 0.25)    # Mo phong ban dem
img_out   = load_image('tablets4.png')        # Ngoai troi — vien thuoc da mau

show_images([img_day, img_night, img_out],
            ['Ban ngay (la mua thu)', 'Mo phong ban dem', 'Ngoai troi (vien thuoc)'],
            cols=3, figsize=(15, 5))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, ttl in zip(axes,
                        [img_day, img_night, img_out],
                        ['Ban ngay', 'Ban dem (mo phong)', 'Ngoai troi']):
    plot_rgb_hist(img, ttl, ax)
plt.suptitle('Histogram RGB tren cung mot khung bieu do', fontsize=13)
plt.tight_layout(); plt.show()

### 2.3 Bài tập 3.3.2 — Histogram kênh H (Hue) trong HSV

In [ ]:
def plot_hue_hist(img, title='Hue Histogram', ax=None):
    H = rgb_to_hsv_numpy(img)[:,:,0]  # [0, 360]
    hist_h, bins = np.histogram(H.ravel(), bins=180, range=(0, 360))
    colors = plt.cm.hsv(np.linspace(0, 1, 180))
    standalone = ax is None
    if standalone:
        _, ax = plt.subplots(figsize=(10, 3))
    ax.bar(bins[:-1], hist_h, width=2.0, color=colors, alpha=0.9)
    ax.set_xlabel('Goc mau H (do)'); ax.set_title(title)
    ax.set_xlim(0, 360); ax.grid(True, alpha=0.3)
    if standalone:
        plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, ttl in zip(axes,
                        [img_day, img_night, img_out],
                        ['Hue — Ban ngay', 'Hue — Ban dem', 'Hue — Ngoai troi']):
    plot_hue_hist(img, ttl, ax)
plt.suptitle('Phan bo Hue (kenh H cua HSV)', fontsize=13)
plt.tight_layout(); plt.show()

### 2.4 Bài tập 3.3.3 — So sánh histogram ban ngày và ban đêm

In [ ]:
x = np.arange(256)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ch, col in zip(axes, range(3), ['red','green','blue']):
    h_day   = np.bincount(img_day  [:,:,ch].ravel(), minlength=256).astype(np.float64)
    h_night = np.bincount(img_night[:,:,ch].ravel(), minlength=256).astype(np.float64)
    ax.plot(x, h_day,   color=col, lw=1.5, label='Ban ngay')
    ax.plot(x, h_night, color=col, lw=1.5, label='Ban dem', linestyle='--', alpha=0.6)
    ax.set_title(f'Kenh {"RGB"[ch]}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('So sanh histogram ban ngay vs ban dem', fontsize=13)
plt.tight_layout(); plt.show()
print('Nhan xet: anh ban dem lenh hoan toan ve phia toi (gia tri thap).')
print('Histogram H cung bi thu hep — phan bo mau sac it da dang hon.')

## 3. Cân bằng histogram ảnh màu

### 3.1 Vấn đề khi cân bằng trực tiếp RGB (§4.1)

Nếu cân bằng độc lập ba kênh $R'=f_R(R),\,G'=f_G(G),\,B'=f_B(B)$, tương quan giữa ba kênh thay đổi mạnh → **Hue bị lệch, màu tự nhiên bị mất.**

**Chiến lược tốt hơn:**
- `Pipeline A:` RGB → HSV → Cân bằng V → RGB
- `Pipeline B:` RGB → Lab → Cân bằng L\* → RGB

### 3.2 Triển khai NumPy + OpenCV (§4.6, §4.7)

In [ ]:
def equalize_channel_numpy(ch):
    '''Can bang histogram mot kenh bang NumPy (§4.6).'''
    hist = np.bincount(ch.ravel(), minlength=256).astype(np.float64)
    cdf  = hist.cumsum()
    cdf_norm = (cdf - cdf.min()) / (cdf.max() - cdf.min() + 1e-12)
    lut = np.floor(255 * cdf_norm).astype(np.uint8)
    return lut[ch]

def equalize_rgb(img):
    '''Can bang doc lap tung kenh RGB — KHONG TOT, chi de so sanh.'''
    return np.stack([equalize_channel_numpy(img[:,:,c]) for c in range(3)], axis=-1)

def equalize_hsv_v(img_rgb):
    '''Pipeline A: can bang kenh V trong HSV (§4.7).'''
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)
    V_eq = cv2.equalizeHist(V)
    return cv2.cvtColor(cv2.merge([H, S, V_eq]), cv2.COLOR_HSV2RGB)

def equalize_lab_l(img_rgb):
    '''Pipeline B: can bang kenh L* trong Lab (§4.7).'''
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, A, Bch = cv2.split(lab)
    L_eq = cv2.equalizeHist(L)
    return cv2.cvtColor(cv2.merge([L_eq, A, Bch]), cv2.COLOR_LAB2RGB)

def clahe_hsv_v(img_rgb, clip=2.0, tile=(8,8)):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)
    cl = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    return cv2.cvtColor(cv2.merge([H, S, cl.apply(V)]), cv2.COLOR_HSV2RGB)

def clahe_lab_l(img_rgb, clip=2.0, tile=(8,8)):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, A, Bch = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    return cv2.cvtColor(cv2.merge([cl.apply(L), A, Bch]), cv2.COLOR_LAB2RGB)

### 3.3 Bài tập 4.8.1 — So sánh 3 cách equalize

In [ ]:
img_dark = load_image('leaves3.png')  # Anh it tuong phan

eq_rgb = equalize_rgb(img_dark)
eq_v   = equalize_hsv_v(img_dark)
eq_l   = equalize_lab_l(img_dark)

show_images([img_dark, eq_rgb, eq_v, eq_l],
            ['Goc (it tuong phan)', 'Eq RGB (lech mau!)', 'Eq V (HSV)', 'Eq L* (Lab)'],
            cols=4, figsize=(20, 5))

### 3.4 Bài tập 4.8.2 — Ảnh phong cảnh và ảnh nền đồng nhất; đánh giá lệch màu

In [ ]:
# Phong canh: leaves.png (nhieu mau sac, nhieu vung)
# Nen dong nhat: seed2.png (hat tren nen kim loai xam)
img_landscape = load_image('leaves.png')
img_uniform   = load_image('seed2.png')

for img, name in [(img_landscape, 'Phong canh (la mua thu)'),
                  (img_uniform,   'Nen dong nhat (hat bi)')]:
    eq_rgb = equalize_rgb(img)
    eq_v   = equalize_hsv_v(img)
    eq_l   = equalize_lab_l(img)
    show_images([img, eq_rgb, eq_v, eq_l],
                [f'{name} (goc)', 'Eq RGB', 'Eq V', 'Eq L*'],
                cols=4, figsize=(20, 4))

print('Nhan xet: Eq RGB gay lech mau ro ret hon tren anh co nen dong nhat')
print('vi moi kenh R,G,B duoc keo dan khac nhau. Anh phong canh co su da dang')
print('mau sac nen su lech mau it an hien hon.')

### 3.5 Bài tập 4.8.3 — CLAHE trên V và L* (thay equalization toàn cục)

In [ ]:
img_test = load_image('leaves3.png')
eq_v  = equalize_hsv_v(img_test)
cl_v  = clahe_hsv_v(img_test)
eq_l  = equalize_lab_l(img_test)
cl_l  = clahe_lab_l(img_test)

show_images([img_test, eq_v, cl_v, eq_l, cl_l],
            ['Goc', 'Eq V (toan cuc)', 'CLAHE V (cuc bo)', 'Eq L*', 'CLAHE L*'],
            cols=5, figsize=(22, 4))
print('CLAHE tranh vung qua sang / qua toi; tot hon anh co chieu sang khong deu.')

### 3.6 Bài tập 4.8.4 — Nhận xét: "Tăng tương phản" vs "Giữ màu tự nhiên"

**Phân tích:**

| Phương pháp | Tăng tương phản | Giữ màu tự nhiên | Phù hợp khi |
|-------------|----------------|------------------|-------------|
| Eq từng kênh RGB | ✅ Mạnh nhất | ❌ Hue/S bị lệch | Không nên dùng cho ảnh màu tự nhiên |
| Eq V (HSV) | ✅ Tốt (V là độ sáng) | ✅ H,S được giữ | Ảnh thiếu sáng, cần tăng độ sáng |
| Eq L\* (Lab) | ✅ Tốt (L\* nhận thức) | ✅✅ Tốt nhất | Ảnh cần cải thiện cho mắt người |
| CLAHE V/L\* | ✅ Tốt cục bộ | ✅✅ Tốt nhất | Ảnh chiếu sáng không đều |

**Mối liên hệ cốt lõi:**

> Tăng tương phản đòi hỏi **phân bố lại giá trị độ sáng**, còn giữ màu tự nhiên đòi hỏi **không thay đổi tương quan giữa các kênh màu**.

Hai mục tiêu này **không xung đột** nếu ta chỉ chỉnh sửa **kênh độ sáng** (V hoặc L\*).

Đó là lý do chiến lược `RGB → HSV → Eq V → RGB` hoặc `RGB → Lab → Eq L\* → RGB` được khuyến nghị:
- V và L\* đại diện cho độ sáng mà **không mang thông tin màu**.
- Chỉnh V hoặc L\* = tăng tương phản mà **không làm méo Hue hay Saturation**.

Tuy nhiên, nếu ảnh quá tối đến mức một số màu biến mất hoàn toàn, dù equalize trên V cũng không thể khôi phục màu đó — đây là **giới hạn vật lý**, không phải lỗi thuật toán.

## 4. Phân ngưỡng tự động trên ảnh màu

### 4.1 Lý thuyết (§5)

Trong ảnh màu, "ngưỡng" thường là:
- Ngưỡng trên **một kênh phù hợp** sau khi đổi hệ màu
- **Miền ngưỡng nhiều chiều**: $H\in[H_1,H_2],\; S\ge S_{min},\; V\ge V_{min}$
- Ngưỡng từ **khoảng cách màu** đến mẫu màu $\mathbf{c}_0$

### 4.2 Otsu từ đầu bằng NumPy (§5.4)

In [ ]:
def otsu_threshold_numpy(channel_uint8):
    '''Otsu tren mot kenh — tra ve (nguong, mask) (§5.4).'''
    hist = np.bincount(channel_uint8.ravel(), minlength=256).astype(np.float64)
    prob = hist / hist.sum()
    omega = np.cumsum(prob)
    mu    = np.cumsum(prob * np.arange(256))
    mu_t  = mu[-1]
    sigma = np.zeros(256)
    valid = (omega > 0) & (omega < 1)
    sigma[valid] = ((mu_t*omega[valid] - mu[valid])**2) / \
                   (omega[valid]*(1.0 - omega[valid]))
    t = int(np.argmax(sigma))
    return t, (channel_uint8 >= t).astype(np.uint8)*255

### 4.3 Bài tập 5.7.1 — Otsu trên R, G, B, S, V, L\*

In [ ]:
def get_channels(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    return {
        'R': img_rgb[:,:,0], 'G': img_rgb[:,:,1], 'B': img_rgb[:,:,2],
        'S': hsv[:,:,1],     'V': hsv[:,:,2],      'L*': lab[:,:,0],
    }

def otsu_all_channels(img_rgb, name=''):
    channels = get_channels(img_rgb)
    masks = []
    labels = []
    for ch_name, ch in channels.items():
        t, mask = otsu_threshold_numpy(ch)
        masks.append(mask)
        labels.append(f'Otsu {ch_name} (t={t})')
    show_images([img_rgb]+masks, [f'Goc ({name})']+labels, cols=4, figsize=(20,10))

img_t = load_image('tablets4.png')
img_s = load_image('seed2.png')
otsu_all_channels(img_t, 'vien thuoc')
otsu_all_channels(img_s, 'hat bi')

### 4.4 Otsu hai bước S rồi V (§5.6) và khoảng cách màu (§5.2.3)

In [ ]:
# === §5.6: Otsu hai buoc S roi V ===
def otsu_two_step_sv(img_rgb):
    '''Pipeline 2: Otsu tren S (loai xam) roi Otsu tren V (§5.6).'''
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H, S, V = cv2.split(hsv)
    _, mask_s = cv2.threshold(S, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    V_masked  = np.where(mask_s > 0, V, 0).astype(np.uint8)
    _, mask_v = cv2.threshold(V_masked, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask_s, mask_v, cv2.bitwise_and(mask_s, mask_v)

# === §5.2.3: Threshold theo khoang cach mau ===
def color_dist_threshold(img_rgb, seed_yx, color_space='lab'):
    '''Otsu tren anh khoang cach mau tu c0 tai seed_yx.'''
    if color_space == 'lab':
        cs = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    elif color_space == 'hsv':
        cs = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    else:
        cs = img_rgb.astype(np.float32)
    sy, sx = seed_yx
    c0   = cs[sy, sx]
    dist = np.sqrt(np.sum((cs - c0)**2, axis=-1))
    d_u8 = (dist / (dist.max()+1e-8) * 255).astype(np.uint8)
    t, mask = otsu_threshold_numpy(d_u8)
    return dist, (d_u8 <= t).astype(np.uint8)*255

# Demo
for img, name in [(img_t, 'vien thuoc'), (img_s, 'hat bi')]:
    ms, mv, mfinal = otsu_two_step_sv(img)
    show_images([img, ms, mv, mfinal],
                [f'Goc ({name})', 'Mask S', 'Mask V (masked)', 'Mask ket hop'],
                cols=4, figsize=(20, 5))

# Color distance demo
img_l = load_image('leaves.png')
H_l, W_l = img_l.shape[:2]
for seed, lbl in [((H_l//5, W_l//5), 'do'), ((H_l*3//4, W_l*3//4), 'vang')]:
    dist, mask = color_dist_threshold(img_l, seed, 'lab')
    vis = img_l.copy()
    cv2.circle(vis, (seed[1],seed[0]), 8, (255,255,255), -1)
    show_images([vis,
                 (dist/dist.max()*255).astype(np.uint8),
                 mask],
                [f'Seed mau {lbl}', 'Khoang cach Lab', 'Mask (Otsu tren khoang cach)'],
                cols=3, figsize=(15, 5))

### 4.5 Multi-level thresholding (§5.2.4)

Thay vì một ngưỡng, dùng nhiều ngưỡng $T_1 < T_2 < \cdots < T_k$ để phân lớp mức sáng hoặc độ bão hòa.

In [ ]:
def otsu_multilevel(channel_uint8, n_thresh=2):
    '''
    Tim n_thresh nguong Otsu toi uu (§5.2.4).
    n_thresh=1: Otsu chuan. n_thresh=2: tim (T1,T2) toi uu (3 lop).
    '''
    hist = np.bincount(channel_uint8.ravel(), minlength=256).astype(np.float64)
    prob = hist / hist.sum()
    omega   = np.cumsum(prob)
    mu_cum  = np.cumsum(prob * np.arange(256))
    mu_tot  = mu_cum[-1]

    if n_thresh == 1:
        sigma = np.zeros(256)
        valid = (omega > 0) & (omega < 1)
        sigma[valid] = ((mu_tot*omega[valid] - mu_cum[valid])**2) / \
                       (omega[valid]*(1 - omega[valid]))
        return [int(np.argmax(sigma))]

    # n_thresh == 2: tim (T1, T2) bang brute-force O(256^2)
    best_sig, best = -1.0, (85, 170)
    for t1 in range(1, 253):
        w0 = omega[t1-1]
        m0 = mu_cum[t1-1]/w0 if w0 > 1e-6 else 0
        for t2 in range(t1+1, 254):
            w1 = omega[t2-1] - omega[t1-1]
            w2 = 1.0 - omega[t2-1]
            if w0<1e-6 or w1<1e-6 or w2<1e-6:
                continue
            m1 = (mu_cum[t2-1]-mu_cum[t1-1])/w1
            m2 = (mu_tot-mu_cum[t2-1])/w2
            sig = w0*(m0-mu_tot)**2 + w1*(m1-mu_tot)**2 + w2*(m2-mu_tot)**2
            if sig > best_sig:
                best_sig, best = sig, (t1, t2)
    return list(best)

def multilevel_segment(img_rgb, channel='V', n_thresh=2):
    '''Phan lop anh mau bang multi-level thresholding tren mot kenh.'''
    channels = get_channels(img_rgb)
    ch = channels[channel]
    thresholds = otsu_multilevel(ch, n_thresh)
    label = np.zeros_like(ch, dtype=np.uint8)
    for i, t in enumerate(thresholds):
        label[ch > t] = i + 1
    return label, thresholds

# Demo: phan lop muc sang tren kenh V, muc bao hoa tren kenh S
for img, name in [(img_t,'vien thuoc'), (img_s,'hat bi')]:
    for ch, title in [('V','phan lop V (do sang)'), ('S','phan lop S (bao hoa)')]:
        label, ths = multilevel_segment(img, ch, n_thresh=2)
        print(f'[{name}] Kenh {ch}: nguong={ths}, so lop={len(ths)+1}')
        show_images([img, colorize_labels(label.astype(np.int32))],
                    [f'Goc ({name})', f'Multi-level {title} T={ths}'],
                    cols=2, figsize=(12, 5))

### 4.6 Adaptive / local thresholding trên ảnh màu (§5.2.5)

Khi chiếu sáng không đồng đều, global threshold thất bại. Giải pháp:
1. Chuyển RGB sang HSV hoặc Lab
2. Lấy kênh V hoặc L\*
3. Áp dụng adaptive thresholding cục bộ

In [ ]:
def adaptive_threshold_color(img_rgb, channel='V', block_size=51, C=5):
    '''
    Adaptive thresholding cuc bo tren mot kenh cua anh mau (§5.2.5).
    block_size: kich thuoc cua so cuc bo (so le).
    C: hang so tru khi tinh nguong cuc bo.
    '''
    block_size = block_size | 1  # Dam bao so le
    channels = get_channels(img_rgb)
    ch = channels[channel]
    return cv2.adaptiveThreshold(ch, 255,
                                  cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, block_size, C)

# So sanh global Otsu vs adaptive tren cac kenh
img_dark2 = simulate_night(load_image('leaves.png'), 0.4)  # Anh co chieu sang khong deu

for img, name in [(img_t,'vien thuoc'), (img_dark2,'la (chieu sang yeu)')]:
    _, mask_global_v = otsu_threshold_numpy(get_channels(img)['V'])
    mask_adapt_v = adaptive_threshold_color(img, 'V', block_size=51)
    _, mask_global_l = otsu_threshold_numpy(get_channels(img)['L*'])
    mask_adapt_l = adaptive_threshold_color(img, 'L*', block_size=51)
    show_images([img, mask_global_v, mask_adapt_v, mask_global_l, mask_adapt_l],
                [f'{name}', 'Otsu V (toan cuc)', 'Adaptive V (cuc bo)',
                 'Otsu L* (toan cuc)', 'Adaptive L* (cuc bo)'],
                cols=5, figsize=(22, 4))

### 4.7 Bài tập 5.7.2 — Pipeline tách trái cây/vật thể màu rõ khỏi nền xám

In [ ]:
def pipeline_colorful_from_gray(img_rgb):
    '''
    Pipeline auto-thresholding: tach vat the co mau ro khoi nen xam (§5.7.2).
    Dua tren Pipeline 1: Otsu tren S (§5.3).
    RGB -> HSV -> Otsu S -> mask vung co mau -> cleanup.
    '''
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    S = hsv[:,:,1]
    _, mask_s = cv2.threshold(S, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Morphology cleanup
    kernel = np.ones((5,5), np.uint8)
    mask_clean = cv2.morphologyEx(mask_s, cv2.MORPH_OPEN,  kernel, iterations=1)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask_s, mask_clean

# tablets4.png: vien thuoc mau tren nen cam (nen co mau nhung it bao hoa hon vien thuoc)
# seed2.png: hat vang-nau tren nen xam kim loai
for img, name in [(img_t,'vien thuoc (mau vs nen cam)'), (img_s,'hat bi (mau vs nen xam)')]:
    ms_raw, ms_clean = pipeline_colorful_from_gray(img)
    result = overlay_mask(img, ms_clean, (0,255,0), 0.4)
    show_images([img, ms_raw, ms_clean, result],
                [f'Goc ({name})', 'Mask S (Otsu)', 'Mask sau cleanup', 'Overlay'],
                cols=4, figsize=(20,5))

### 4.8 Bài tập 5.7.4 — Khi nào Otsu trên S tốt hơn V và ngược lại?

**Phân tích:**

| Kênh | Đại diện cho | Tốt khi nào |
|------|--------------|-------------|
| **S** (Saturation) | Mức độ tinh khiết màu sắc | Vật thể có màu rõ ràng trên nền **xám/trắng/đen** hoặc nền có màu nhạt |
| **V** (Value) | Độ sáng cực đại | Tách vùng **sáng/tối**, ví dụ chữ đen trên giấy trắng, hoặc vật thể sáng trên nền tối |

**Otsu trên S tốt hơn V khi:**
- Đối tượng có màu **sắc nét, bão hòa cao** (trái cây đỏ, biển báo xanh/đỏ)
- Nền là màu **trung tính** (xám, trắng, đen) có S thấp
- Đối tượng và nền có **độ sáng (V) tương tự** nhau → Otsu trên V thất bại

**Otsu trên V tốt hơn S khi:**
- Cần tách vùng **sáng/tối** bất kể màu sắc
- Đối tượng và nền **đều có màu sắc** nhưng khác nhau về độ sáng
- Ảnh có các vùng phản xạ sáng hoặc bóng tối

**Bài toán cụ thể:**
- `seed2.png`: hạt bí (màu vàng-nâu) trên nền kim loại (xám trung tính) → S phân biệt tốt
- `tablets4.png`: viên thuốc đa màu trên nền cam → cả S và H đều hữu ích; V ít phân biệt được vì nền và thuốc có V tương tự
- Ảnh ban đêm mô phỏng → V thất bại (tất cả tối) nhưng H/S vẫn giữ thông tin màu nếu chụp đúng ISO

**Kết luận:** Không có kênh nào "tốt nhất" tuyệt đối — lựa chọn phụ thuộc vào **tính chất phân biệt** của đối tượng vs nền trong bài toán cụ thể. Pipeline Otsu hai bước S→V là **chiến lược thực dụng** tốt nhất trong phần lớn trường hợp.

### 4.9 Bài tập 5.6 — Trích xuất vùng da người (Skin detection)

Dựa trên không gian màu HSV và YCbCr. Vùng da người thường nằm trong dải:
- HSV: $H \in [0, 20]$, $S \in [0.2, 0.6]$, $V \in [0.4, 1.0]$.
- YCbCr: $Y > 80$, $77 < Cb < 127$, $133 < Cr < 173$.

In [ ]:
def detect_skin(img_rgb):
    # Chuyen sang HSV
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    # Nguong HSV cho da nguoi (OpenCV H: 0-180, S: 0-255, V: 0-255)
    lower_hsv = np.array([0, 48, 80], dtype=np.uint8)
    upper_hsv = np.array([20, 255, 255], dtype=np.uint8)
    mask_hsv = cv2.inRange(hsv, lower_hsv, upper_hsv)

    # Chuyen sang YCbCr
    ycbcr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)
    # Nguong YCbCr (Y, Cr, Cb)
    lower_ycbcr = np.array([80, 133, 77], dtype=np.uint8)
    upper_ycbcr = np.array([255, 173, 127], dtype=np.uint8)
    mask_ycbcr = cv2.inRange(ycbcr, lower_ycbcr, upper_ycbcr)

    # Ket hop
    mask_final = cv2.bitwise_and(mask_hsv, mask_ycbcr)
    return mask_hsv, mask_ycbcr, mask_final

# Demo tren mot vung co mau da (lay tu tablets4 co vung mau cam/hong)
mask_h, mask_y, mask_f = detect_skin(img_t)
show_images([img_t, mask_h, mask_y, mask_f],
            ['Goc', 'Mask HSV', 'Mask YCbCr', 'Mask Ket hop'],
            cols=4, figsize=(20, 5))

## 6. Các bộ lọc trên ảnh màu (Filtering)


In [ ]:
def smoothing_channels(img_rgb, ks=5):
    '''Lam min doc lap 3 kenh RGB.'''
    return np.stack([cv2.GaussianBlur(img_rgb[:,:,c], (ks,ks), 0) for c in range(3)], axis=-1)

def smoothing_hsv_v(img_rgb, ks=5):
    '''Lam min chi tren kenh V trong HSV.'''
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    hsv[:,:,2] = cv2.GaussianBlur(hsv[:,:,2], (ks,ks), 0)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

img_test = load_image('leaves.png')
smooth_rgb = smoothing_channels(img_test, 15)
smooth_v   = smoothing_hsv_v(img_test, 15)

show_images([img_test, smooth_rgb, smooth_v],
            ['Goc', 'Smooth RGB (all channels)', 'Smooth V (HSV)'],
            cols=3, figsize=(15, 5))

### 5.2 Làm sắc nét (Sharpening)
Tương tự, làm sắc nét thường dùng **Unsharp Masking** trên kênh độ sáng để tránh nhiễu màu sắc (color artifacts).

In [ ]:
def sharpen_lab_l(img_rgb, sigma=1.0, amount=1.5):
    '''Lam sac net tren kenh L* cua Lab.'''
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L = lab[:,:,0].astype(np.float32)
    blurred = cv2.GaussianBlur(L, (0,0), sigma)
    sharpened = L + amount * (L - blurred)
    lab[:,:,0] = np.clip(sharpened, 0, 255).astype(np.uint8)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

img_blur = cv2.GaussianBlur(img_test, (7,7), 2)
sharp = sharpen_lab_l(img_blur)
show_images([img_test, img_blur, sharp],
            ['Goc', 'Bi mo (Gaussian)', 'Sac net (Unsharp L*)'],
            cols=3, figsize=(15, 5))

## 6. Xây dựng các thuật toán kiểu Canny cho ảnh màu (§6)

### 6.1 Ba cách xây dựng Color Canny (§6.2)

- **Cách 1:** Canny trên từng kênh R, G, B rồi hợp nhất (OR/max)
- **Cách 2:** Gradient vector màu — xây dựng tensor cấu trúc $\mathbf{G}$, lấy biên từ trị riêng lớn nhất
- **Cách 3:** Chuyển sang không gian màu phù hợp (L\*, S) rồi Canny

### 6.2 Triển khai (§6.3, §6.4)

In [ ]:
SOBEL_KX = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
SOBEL_KY = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)

# === Cach 1: Multi-channel Canny ===
def multichannel_canny(img_rgb, lo=50, hi=150, sigma=1.5):
    '''Canny tren R,G,B roi hop nhat bang max (§6.2.1).'''
    ks = int(6*sigma+1)|1
    blurred = cv2.GaussianBlur(img_rgb, (ks,ks), sigma)
    edges = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
    for c in range(3):
        edges = np.maximum(edges, cv2.Canny(blurred[:,:,c], lo, hi))
    return edges

# === Cach 2: Gradient vector mau (§6.4) ===
def color_gradient_magnitude(img_float, kx, ky):
    '''
    Do manh bien tu gradient vector mau (§6.4).
    img_float: H x W x 3 float32 in [0,1].
    '''
    Ix = np.stack([cv2.filter2D(img_float[:,:,c], -1, kx) for c in range(3)], axis=-1)
    Iy = np.stack([cv2.filter2D(img_float[:,:,c], -1, ky) for c in range(3)], axis=-1)
    gxx = np.sum(Ix*Ix, axis=-1)
    gyy = np.sum(Iy*Iy, axis=-1)
    gxy = np.sum(Ix*Iy, axis=-1)
    tmp = np.sqrt((gxx-gyy)**2 + 4*gxy**2)
    mag   = np.sqrt((gxx+gyy+tmp)/2.0)
    theta = 0.5*np.arctan2(2*gxy, gxx-gyy+1e-12)
    return mag, theta

# === Cach 3: Canny tren kenh cu the ===
def canny_on_channel(img_rgb, channel='gray', lo=50, hi=150, sigma=1.5):
    ks = int(6*sigma+1)|1
    if   channel == 'gray': ch = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    elif channel == 'L':    ch = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0]
    elif channel == 'S':    ch = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)[:,:,1]
    elif channel == 'V':    ch = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)[:,:,2]
    else: raise ValueError(f'Unknown channel: {channel}')
    return cv2.Canny(cv2.GaussianBlur(ch,(ks,ks),sigma), lo, hi)

### 5.3 Bài tập 6.5.1 — Multi-channel Canny vs Canny grayscale

In [ ]:
img = load_image('leaves.png')
e_gray  = canny_on_channel(img, 'gray')
e_multi = multichannel_canny(img)
show_images([img, e_gray, e_multi],
            ['Anh goc', 'Canny Grayscale', 'Multi-channel Canny (OR)'],
            cols=3, figsize=(15, 5))
print('Multi-channel nhan ra bien giua la co mau khac nhau nhung do sang giong nhau.')

### 5.4 Bài tập 6.5.2 — Gradient vector màu (dừng ở magnitude map)

In [ ]:
img = load_image('leaves.png')
img_f = img.astype(np.float32)/255.0
gray_f = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0

mag_color, theta_color = color_gradient_magnitude(
    cv2.GaussianBlur(img_f,(5,5),1.5), SOBEL_KX, SOBEL_KY)

gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0)
gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1)
mag_gray = np.sqrt(gx**2 + gy**2)

def norm255(x): return (x/(x.max()+1e-8)*255).astype(np.uint8)

show_images([img, norm255(mag_gray), norm255(mag_color)],
            ['Anh goc', 'Gradient Sobel (Grayscale)', 'Gradient vector mau (§6.4)'],
            cols=3, figsize=(15, 5))
print('Gradient mau manh hon o bien giua la co mau khac nhau (grayscale bo sot).')

### 5.5 Bài tập 6.5.3 — "Mini color Canny" có NMS (vectorized)

In [ ]:
def nms_vectorized(mag, theta):
    '''Non-maximum suppression vectorized bang NumPy (nhanh hon vong lap).'''
    H, W  = mag.shape
    mp    = np.pad(mag, 1, mode='edge')
    angle = np.degrees(theta) % 180
    q1 = np.zeros_like(mag); q2 = np.zeros_like(mag)
    # 0 do: ngang
    m = (angle < 22.5)|(angle >= 157.5)
    q1[m] = mp[1:-1, 2:][m];   q2[m] = mp[1:-1, :-2][m]
    # 45 do: duong cheo
    m = (angle >= 22.5)&(angle < 67.5)
    q1[m] = mp[:-2, 2:][m];    q2[m] = mp[2:, :-2][m]
    # 90 do: doc
    m = (angle >= 67.5)&(angle < 112.5)
    q1[m] = mp[:-2, 1:-1][m];  q2[m] = mp[2:, 1:-1][m]
    # 135 do: duong cheo nguoc
    m = (angle >= 112.5)&(angle < 157.5)
    q1[m] = mp[:-2, :-2][m];   q2[m] = mp[2:, 2:][m]
    result = mag.copy()
    result[(mag < q1)|(mag < q2)] = 0
    return result

def mini_color_canny(img_rgb, lo_ratio=0.1, hi_ratio=0.3, sigma=1.5):
    '''Mini color Canny: smoothing -> color gradient -> NMS -> double threshold.'''
    ks = int(6*sigma+1)|1
    img_f = cv2.GaussianBlur(img_rgb.astype(np.float32)/255.0, (ks,ks), sigma)
    mag, theta = color_gradient_magnitude(img_f, SOBEL_KX, SOBEL_KY)
    mag_n = mag / (mag.max()+1e-8)
    mag_nms = nms_vectorized(mag_n, theta)
    # Double threshold
    lo, hi = lo_ratio * mag_nms.max(), hi_ratio * mag_nms.max()
    strong = (mag_nms >= hi).astype(np.uint8)*255
    weak   = ((mag_nms >= lo)&(mag_nms < hi)).astype(np.uint8)*128
    edge   = strong | weak
    return edge, norm255(mag_n), norm255(mag_nms)

img = load_image('tablets4.png')
edge_mini, mag_vis, nms_vis = mini_color_canny(img)
show_images([img, mag_vis, nms_vis, edge_mini],
            ['Anh goc', 'Color gradient magnitude', 'Sau NMS', 'Mini color Canny'],
            cols=4, figsize=(20, 5))

### 5.6 Bài tập 6.5.4 — Ảnh mà grayscale Canny bỏ sót biên nhưng color-aware phát hiện được

In [ ]:
for img, name in [(load_image('leaves.png'),'La mua thu'),
                  (load_image('tablets4.png'),'Vien thuoc')]:
    e_gray  = canny_on_channel(img, 'gray')
    e_l     = canny_on_channel(img, 'L',  lo=50, hi=130)
    e_s     = canny_on_channel(img, 'S',  lo=30, hi=100)
    e_multi = multichannel_canny(img)
    # Bien bị bỏ sót bởi grayscale nhưng phát hiện bởi multi-channel
    missed  = cv2.bitwise_and(e_multi, cv2.bitwise_not(e_gray))
    show_images([img, e_gray, e_s, e_multi, missed],
                [name, 'Canny Gray (bo sot bien mau)', 'Canny S', 'Multi-channel',
                 'Bien GRAY bo sot / COLOR phat hien'],
                cols=3, figsize=(15,10))

## 7. Phân đoạn dựa trên vùng cho ảnh màu (§7)

Tiêu chuẩn đồng nhất bên trong vùng: $\|\mathbf{c}(x,y)-\boldsymbol{\mu}_R\|\leq T$

### 7.1 Region Growing trong Lab (§7.4)

In [ ]:
def region_growing_lab(img_rgb, seed_yx, thresh=15.0):
    '''Region growing trong khong gian Lab (§7.4).'''
    lab  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    H, W, _ = lab.shape
    visited = np.zeros((H,W), dtype=bool)
    mask    = np.zeros((H,W), dtype=np.uint8)
    sy, sx  = seed_yx
    q = deque([(sy,sx)])
    mean_c, count = lab[sy,sx].copy(), 1
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)]
    while q:
        y, x = q.popleft()
        if visited[y,x]: continue
        visited[y,x] = True
        if np.linalg.norm(lab[y,x]-mean_c) <= thresh:
            mask[y,x] = 255
            mean_c = (mean_c*count + lab[y,x])/(count+1)
            count += 1
            for dy,dx in nbrs:
                ny,nx = y+dy, x+dx
                if 0<=ny<H and 0<=nx<W and not visited[ny,nx]:
                    q.append((ny,nx))
    return mask

def region_growing_rgb(img_rgb, seed_yx, thresh=15.0):
    '''Region growing trong RGB (de so sanh).'''
    img_f = img_rgb.astype(np.float32)
    H,W,_ = img_f.shape
    visited = np.zeros((H,W), dtype=bool)
    mask = np.zeros((H,W), dtype=np.uint8)
    sy,sx = seed_yx
    q = deque([(sy,sx)])
    mean_c, count = img_f[sy,sx].copy(), 1
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)]
    while q:
        y,x = q.popleft()
        if visited[y,x]: continue
        visited[y,x] = True
        if np.linalg.norm(img_f[y,x]-mean_c) <= thresh:
            mask[y,x] = 255
            mean_c = (mean_c*count+img_f[y,x])/(count+1)
            count += 1
            for dy,dx in nbrs:
                ny,nx = y+dy, x+dx
                if 0<=ny<H and 0<=nx<W and not visited[ny,nx]:
                    q.append((ny,nx))
    return mask

### 6.2 Bài tập 7.5.1 — Region growing RGB vs Lab (so sánh độ ổn định)

In [ ]:
img = load_image('tablets4.png')
H, W = img.shape[:2]
seeds = [(H//4, W//4, (0,200,0)), (H//2, W//2, (200,0,0)), (H*3//4, W*3//4, (0,0,200))]

fig, axes = plt.subplots(2, len(seeds)+1, figsize=(20, 10))
for row, (fn, space) in enumerate([(region_growing_rgb,'RGB'), (region_growing_lab,'Lab')]):
    axes[row,0].imshow(img)
    for sy,sx,col in seeds:
        axes[row,0].plot(sx,sy,'x',color=np.array(col)/255.,ms=14,mew=3)
    axes[row,0].set_title(f'Seeds ({space})'); axes[row,0].axis('off')
    for ci,(sy,sx,col) in enumerate(seeds):
        mask = fn(img,(sy,sx),thresh=20.0)
        axes[row,ci+1].imshow(overlay_mask(img,mask,col))
        axes[row,ci+1].set_title(f'{space} seed ({sy},{sx}) n={mask.sum()//255}')
        axes[row,ci+1].axis('off')
plt.suptitle('Region Growing: RGB vs Lab (thresh=20)', fontsize=13)
plt.tight_layout(); plt.show()
print('Lab thuong on dinh hon: it bi anh huong boi su khac biet do sang.')

### 6.3 Bài tập 7.5.2 — K-means màu vs màu + (x,y)

In [ ]:
def kmeans_color(img_rgb, k=5, use_spatial=False, sw=0.15, seed=42):
    '''K-means tren Lab (+ toa do neu use_spatial=True) (§7.2.3).'''
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    H, W, _ = lab.shape
    feat = lab.reshape(-1,3) / np.array([100.,128.,128.])
    if use_spatial:
        yy, xx = np.mgrid[0:H, 0:W]
        feat = np.concatenate(
            [feat,
             (yy.reshape(-1,1)/max(H-1,1))*sw,
             (xx.reshape(-1,1)/max(W-1,1))*sw], axis=1)
    rng = np.random.default_rng(seed)
    centers = feat[rng.choice(feat.shape[0], k, replace=False)].copy()
    labels  = np.zeros(feat.shape[0], dtype=np.int32)
    for _ in range(30):
        d = np.sum((feat[:,None,:]-centers[None,:,:])**2, axis=2)
        new_labels = np.argmin(d, axis=1)
        new_c = centers.copy()
        for c in range(k):
            pts = feat[new_labels==c]
            if len(pts): new_c[c] = pts.mean(axis=0)
        if np.linalg.norm(new_c-centers) < 1e-3: labels=new_labels; break
        labels, centers = new_labels, new_c
    return labels.reshape(H,W)

for img, name in [(load_image('tablets4.png'),'Vien thuoc'), (load_image('leaves.png'),'La')]:
    k = 6
    lc = kmeans_color(img, k=k)
    ls = kmeans_color(img, k=k, use_spatial=True, sw=0.3)
    show_images([img, colorize_labels(lc), colorize_labels(ls)],
                [f'{name} (goc)', f'K-means Lab (k={k})', f'K-means Lab+XY (k={k})'],
                cols=3, figsize=(15,5))
print('Them toa do (x,y) giu tinh lien thong khong gian, tranh phan cum roi rac.')

### 6.4 Bài tập 7.5.3 — Pipeline phân đoạn lá cây: region growing + morphology

In [ ]:
img_leaves = load_image('leaves.png')
H_l, W_l = img_leaves.shape[:2]
leaf_seeds = [
    (H_l//5,     W_l//5,     (230, 60, 60)),   # La do
    (H_l//2,     W_l//10,    (230,150, 50)),    # La cam
    (H_l*4//5,   W_l*3//4,   (220,210, 50)),    # La vang
    (H_l//3,     W_l*2//3,   (140, 50,140)),    # La tim
]
kernel5 = np.ones((5,5), np.uint8)

results, lbls = [img_leaves], ['Goc + seeds']
all_masks = []
for sy,sx,col in leaf_seeds:
    raw   = region_growing_lab(img_leaves, (sy,sx), thresh=13.0)
    clean = cv2.morphologyEx(raw,   cv2.MORPH_CLOSE, kernel5, iterations=2)
    clean = cv2.morphologyEx(clean, cv2.MORPH_OPEN,  kernel5, iterations=1)
    results.append(overlay_mask(img_leaves, clean, col))
    lbls.append(f'seed({sy},{sx})')
    all_masks.append(clean)

combined = np.zeros(img_leaves.shape[:2], np.uint8)
for m in all_masks: combined = np.maximum(combined, m)
results.append(overlay_mask(img_leaves, combined, (100,200,100)))
lbls.append('Tat ca vung la')

show_images(results, lbls, cols=3, figsize=(15,15))

### 6.5 Watershed với marker trên ảnh màu (§7.2.4)

Pipeline:
1. Chuyển sang gradient màu hoặc channel phù hợp
2. Tạo marker (sure fg, sure bg, unknown)
3. Chạy `cv2.watershed`

In [ ]:
def watershed_color(img_rgb, sigma=1.5, dist_ratio=0.4):
    '''
    Watershed voi marker tren anh mau (§7.2.4).
    Dung kenh S de xay dung marker (vung co mau ro = foreground).
    '''
    ks = int(6*sigma+1)|1
    blurred = cv2.GaussianBlur(img_rgb, (ks,ks), sigma)
    # Dung Otsu tren S de tach foreground
    hsv = cv2.cvtColor(blurred, cv2.COLOR_RGB2HSV)
    S   = hsv[:,:,1]
    _, binary = cv2.threshold(S, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Morphology
    kernel = np.ones((3,3), np.uint8)
    opening  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel, iterations=2)
    sure_bg  = cv2.dilate(opening, kernel, iterations=3)
    dist_map = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist_map, dist_ratio*dist_map.max(), 255, 0)
    sure_fg  = sure_fg.astype(np.uint8)
    unknown  = cv2.subtract(sure_bg, sure_fg)
    # Marker labeling
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    markers_ws = cv2.watershed(img_bgr, markers.copy())
    result = img_rgb.copy()
    result[markers_ws == -1] = [255, 0, 0]  # Bien do
    return markers_ws, result, opening, sure_fg

for img, name in [(load_image('tablets4.png'),'vien thuoc'),
                  (load_image('seed2.png'),   'hat bi')]:
    markers, result, opening, sure_fg = watershed_color(img)
    n_obj = markers.max() - 1
    show_images([img, opening, sure_fg, result],
                [f'Goc ({name})', 'Opening (binary)', 'Sure FG', f'Watershed ({n_obj} obj, bien do)'],
                cols=4, figsize=(20,5))

### 6.6 Split-and-merge theo độ đồng nhất màu (§7.2.2)

**Tiêu chuẩn dừng split:** phương sai màu Lab trong vùng $< T_{split}$

$$\text{color\_var}(R)=\frac{1}{3}\sum_{c}\text{Var}\bigl(\{c(x,y)\}_{(x,y)\in R}\bigr)$$

**Tiêu chuẩn merge:** khoảng cách Lab giữa mean của hai vùng lân cận $\leq T_{merge}$

In [ ]:
def split_merge_color(img_rgb, min_size=16, split_thresh=200.0, merge_thresh=20.0):
    '''
    Split-and-merge cho anh mau dua tren phuong sai mau Lab (§7.2.2, §7.5.4).
    '''
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    H, W, _ = lab.shape

    # === SPLIT PHASE: Quadtree recursion ===
    rects = []
    def color_var(x,y,h,w):
        region = lab[x:x+h, y:y+w].reshape(-1,3)
        return float(np.mean(np.var(region, axis=0)))

    def split(x,y,h,w):
        if h<=min_size or w<=min_size or color_var(x,y,h,w)<split_thresh:
            rects.append((x,y,h,w)); return
        h2,w2 = h//2, w//2
        if h2==0 or w2==0: rects.append((x,y,h,w)); return
        split(x,   y,   h2,   w2);  split(x,   y+w2, h2,   w-w2)
        split(x+h2,y,   h-h2, w2);  split(x+h2,y+w2, h-h2, w-w2)

    split(0,0,H,W)
    n = len(rects)

    # === MERGE PHASE: Union-Find ===
    means = np.array([
        lab[x:x+h,y:y+w].reshape(-1,3).mean(axis=0)
        for x,y,h,w in rects], dtype=np.float32)

    parent = list(range(n))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(i,j):
        pi,pj = find(i),find(j)
        if pi!=pj: parent[pi]=pj
    def adjacent(i,j):
        x1,y1,h1,w1 = rects[i]; x2,y2,h2,w2 = rects[j]
        th = (y1+w1==y2 or y2+w2==y1) and not(x1>=x2+h2 or x2>=x1+h1)
        tv = (x1+h1==x2 or x2+h2==x1) and not(y1>=y2+w2 or y2>=y1+w1)
        return th or tv

    for i in range(n):
        for j in range(i+1,n):
            if adjacent(i,j) and np.linalg.norm(means[i]-means[j])<=merge_thresh:
                union(i,j)

    # Build label map
    lmap = np.zeros((H,W), dtype=np.int32)
    r2l  = {}; ctr = [1]
    for idx,(x,y,h,w) in enumerate(rects):
        root = find(idx)
        if root not in r2l: r2l[root]=ctr[0]; ctr[0]+=1
        lmap[x:x+h,y:y+w] = r2l[root]
    return lmap, len(r2l), rects

### 6.7 Bài tập 7.5.4 — Tiêu chuẩn dừng split-and-merge theo phương sai màu

In [ ]:
for img, name in [(load_image('tablets4.png'),'vien thuoc'),
                  (load_image('leaves.png'),  'la mua thu')]:
    # Demo voi cac nguong khac nhau
    lmap1, k1, _ = split_merge_color(img, min_size=16, split_thresh=150, merge_thresh=15)
    lmap2, k2, _ = split_merge_color(img, min_size=16, split_thresh=300, merge_thresh=25)
    print(f'[{name}] T_split=150/T_merge=15: {k1} vung | T_split=300/T_merge=25: {k2} vung')
    show_images([img, colorize_labels(lmap1), colorize_labels(lmap2)],
                [f'Goc ({name})',
                 f'Split-merge (split=150,merge=15) -> {k1} vung',
                 f'Split-merge (split=300,merge=25) -> {k2} vung'],
                cols=3, figsize=(15,5))

print('Tieu chuan dung:')
print('  SPLIT: phuong sai Lab trung binh cua vung < split_thresh -> khong chia tiep')
print('  MERGE: khoang cach Lab giua mean cua hai vung lan can <= merge_thresh -> gop lai')
print('  Tieu chi nay giup phan doan tot hon grayscale vi tinh den toan bo thong tin mau.')

## 8. Phân đoạn dựa trên biên cho ảnh màu (§8)

**Ba tầng:** edge detection → contour formation → region extraction

**Thách thức với ảnh màu:**
- Biên có thể xuất hiện do khác biệt màu dù độ sáng gần như nhau
- Biên từ các kênh khác nhau có thể không trùng hoàn toàn

### 8.1 Code minh họa edge-to-region (§8.4)

In [ ]:
def edge_to_region(img_rgb, lo=80, hi=160, close_iter=1, min_area=200):
    '''
    Pipeline (§8.4): bien Lab Canny -> dong kin -> contour -> fill.
    '''
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    # Canny tren L*, a*, b*
    eL = cv2.Canny(lab[:,:,0], lo,    hi)
    eA = cv2.Canny(lab[:,:,1], lo//2, hi//2)
    eB = cv2.Canny(lab[:,:,2], lo//2, hi//2)
    edge = np.maximum(eL, np.maximum(eA,eB)).astype(np.uint8)
    # Dong bien
    kernel = np.ones((3,3),np.uint8)
    closed = cv2.morphologyEx(edge, cv2.MORPH_CLOSE, kernel, iterations=close_iter)
    # Contour + fill
    contours,_ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours   = [c for c in contours if cv2.contourArea(c)>=min_area]
    mask = np.zeros(edge.shape, np.uint8)
    cv2.drawContours(mask, contours, -1, 255, cv2.FILLED)
    result = img_rgb.copy()
    cv2.drawContours(result, contours, -1, (0,255,0), 2)
    return edge, closed, mask, result, contours

### 7.2 Bài tập 8.5.1 — Canny L\*, S, color gradient; so sánh contour

In [ ]:
for img, name in [(load_image('tablets4.png'),'vien thuoc'),
                  (load_image('leaves.png'),  'la mua thu')]:
    e_gray  = canny_on_channel(img, 'gray')
    e_l     = canny_on_channel(img, 'L')
    e_s     = canny_on_channel(img, 'S', lo=30, hi=100)
    e_multi = multichannel_canny(img)
    img_f   = img.astype(np.float32)/255.0
    mag,_   = color_gradient_magnitude(img_f, SOBEL_KX, SOBEL_KY)
    _, e_vec = cv2.threshold(norm255(mag), 0,255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    show_images([img, e_gray, e_l, e_s, e_multi, e_vec],
                [name,'Canny Gray','Canny L*','Canny S','Multi-channel','Vector Gradient'],
                cols=3, figsize=(15,10))

### 7.3 Bài tập 8.5.2 — Pipeline biên màu → contour → mask đối tượng

In [ ]:
for img, name, lo, hi in [
    (load_image('tablets4.png'), 'vien thuoc', 60, 140),
    (load_image('leaves.png'),   'la mua thu', 50, 120),
]:
    edge, closed, mask, result, ctrs = edge_to_region(img, lo=lo, hi=hi,
                                                       close_iter=2, min_area=300)
    print(f'[{name}] So contour: {len(ctrs)}')
    show_images([img, edge, closed, mask, result],
                [f'{name}', 'Bien Lab', 'Dong bien', 'Fill mask', f'Contour ({len(ctrs)})'],
                cols=3, figsize=(15,10))

### 7.4 Bài tập 8.5.3 — Ảnh có độ sáng giống nhau nhưng màu khác nhau

In [ ]:
for img, name in [(load_image('leaves.png'),'La mua thu'),
                  (load_image('tablets4.png'),'Vien thuoc')]:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    e_gray  = cv2.Canny(cv2.GaussianBlur(gray,(5,5),1.5), 50, 130)
    e_l     = canny_on_channel(img, 'L')
    e_s     = canny_on_channel(img, 'S', lo=30, hi=100)
    # Tinh % bien phat hien bo sung boi color-aware
    extra_s = cv2.bitwise_and(e_s, cv2.bitwise_not(e_gray))
    pct = extra_s.sum()/(e_gray.sum()+1e-6)*100
    show_images([img, gray, e_gray, e_s, extra_s],
                [f'{name}','Grayscale','Canny Gray',
                 'Canny S (bien mau)',
                 f'Bo sung boi S ({pct:.0f}% so voi gray)'],
                cols=3, figsize=(15,10))

print('Vung la co mau khac nhau (do/vang/tim) nhung do sang tuong tu -> grayscale bo sot.')
print('Kenh S (bao hoa mau) nhan ra cac vung bien mau nay.')

### 7.5 Active contour / Snake với color edge map (§8.2 H3, H4)

**H3 — Active contour màu (Snake):**
$$E_{image} = w_1|\nabla L^*| + w_2|\nabla S| + w_3|\nabla\text{color-dist}|$$

**H4 — Hàm dừng geodesic:**
$$g(x,y) = \frac{1}{1+\alpha M(x,y)^2}$$
trong đó $M(x,y)$ là độ mạnh biên màu. Contour dừng lại tại vùng $g$ thấp (biên màu mạnh).

In [ ]:
def geodesic_stopping_fn(img_rgb, alpha=200.0, sigma=1.5):
    '''Ham dung geodesic g(x,y)=1/(1+alpha*M^2) (§8.2 H4).'''
    ks = int(6*sigma+1)|1
    img_f = cv2.GaussianBlur(img_rgb.astype(np.float32)/255.0,(ks,ks),sigma)
    mag,_ = color_gradient_magnitude(img_f, SOBEL_KX, SOBEL_KY)
    mag_n = mag/(mag.max()+1e-8)
    g     = 1.0/(1.0 + alpha*mag_n**2)
    return g, mag_n


def morphological_chan_vese_color(img_rgb, init_mask, n_iter=60, smoothing=2):
    '''
    Morphological Chan-Vese active contour cho anh mau (§8.2 H3).
    Luc ngoai = khoang cach Lab den mean trong/ngoai contour.
    '''
    lab    = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    mask   = (init_mask > 0).copy()
    kernel = np.ones((3,3),np.uint8)
    for _ in range(n_iter):
        inside  = lab[ mask]
        outside = lab[~mask]
        if len(inside)<1 or len(outside)<1: break
        c1 = inside.mean(axis=0)
        c2 = outside.mean(axis=0)
        # Pixel thuoc vung 1 (inside) neu khoang cach den c1 nho hon c2
        d1   = np.sum((lab-c1)**2, axis=-1)
        d2   = np.sum((lab-c2)**2, axis=-1)
        nm_u8 = (d1<d2).astype(np.uint8)*255
        for _ in range(smoothing):
            nm_u8 = cv2.morphologyEx(nm_u8, cv2.MORPH_OPEN,  kernel)
            nm_u8 = cv2.morphologyEx(nm_u8, cv2.MORPH_CLOSE, kernel)
        mask = nm_u8 > 0
    return mask


def snake_color(img_rgb, center_yx, radius=50, n_pts=80,
                n_iter=100, alpha=300.0, step=1.5, sigma=2.0):
    '''
    Snake: contour da giac, moi diem di chuyen nguc chieu gradient g
    (tien den vung bien mau manh = g thap).
    '''
    H, W = img_rgb.shape[:2]
    g, mag = geodesic_stopping_fn(img_rgb, alpha=alpha, sigma=sigma)
    gx = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    cy, cx = center_yx
    angles = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
    pts = np.stack([cy+radius*np.sin(angles),
                    cx+radius*np.cos(angles)], axis=-1).astype(np.float32)
    history = [pts.astype(np.int32).copy()]
    for it in range(n_iter):
        new_pts = pts.copy()
        for i,(py,px) in enumerate(pts):
            iy = int(np.clip(py,1,H-2)); ix = int(np.clip(px,1,W-2))
            dy = -gy[iy,ix]; dx = -gx[iy,ix]
            n_ = np.sqrt(dy**2+dx**2)+1e-8
            new_pts[i,0] = np.clip(py+step*dy/n_,1,H-2)
            new_pts[i,1] = np.clip(px+step*dx/n_,1,W-2)
        pts = new_pts
        if it in (n_iter//4, n_iter//2, 3*n_iter//4, n_iter-1):
            history.append(pts.astype(np.int32).copy())
    return pts.astype(np.int32), history, g, mag

### 7.6 Bài tập 8.5.4 — Mini-project: Active contour với color edge map

In [ ]:
# === Demo 1: Ham dung geodesic g(x,y) ===
img = load_image('tablets4.png')
g, mag = geodesic_stopping_fn(img, alpha=200.0)
show_images([img, norm255(mag), (g*255).astype(np.uint8)],
            ['Anh goc', 'Do manh bien mau M(x,y)',
             'Ham dung geodesic g(x,y) — thap tren bien, cao o vung dong nhat'],
            cols=3, figsize=(15,5))

# === Demo 2: Snake tien hoa theo g ===
img = load_image('tablets4.png')
H, W = img.shape[:2]
final_pts, history, g, mag = snake_color(img, center_yx=(H//2,W//2),
                                          radius=min(H,W)//4,
                                          n_iter=80, alpha=300.0, step=1.5)
fig, axes = plt.subplots(1, 3, figsize=(18,6))
for ax, title in zip(axes, ['Trang thai dau','Qua trinh tien hoa','Ket qua cuoi']):
    ax.imshow(img); ax.axis('off'); ax.set_title(title)
colors_ev = plt.cm.cool(np.linspace(0,1,len(history)))
for hi_i, pts_h in enumerate(history):
    closed = np.vstack([pts_h, pts_h[0]])
    axes[1].plot(closed[:,1], closed[:,0], color=colors_ev[hi_i], lw=1.5, alpha=0.7)
closed_f = np.vstack([history[0], history[0][0]])
axes[0].plot(closed_f[:,1], closed_f[:,0], 'y-', lw=2)
closed_l = np.vstack([final_pts, final_pts[0]])
axes[2].plot(closed_l[:,1], closed_l[:,0], 'lime', lw=2)
plt.suptitle('Snake voi color edge map (g(x,y)) — §8.2 H4', fontsize=13)
plt.tight_layout(); plt.show()

# === Demo 3: Morphological Chan-Vese cho anh mau ===
img = load_image('tablets4.png')
H, W = img.shape[:2]
# Khoi tao contour hinh tron o trung tam
yy, xx = np.mgrid[0:H, 0:W]
cy, cx = H//2, W//2
init_mask = ((yy-cy)**2 + (xx-cx)**2 <= (min(H,W)//4)**2).astype(np.uint8)*255
result_mask = morphological_chan_vese_color(img, init_mask, n_iter=50, smoothing=2)
show_images([img,
             overlay_mask(img, init_mask,   (255,255,0), 0.3),
             overlay_mask(img, result_mask.astype(np.uint8)*255, (0,255,0), 0.4)],
            ['Anh goc',
             'Contour khoi tao (hinh tron)',
             'Sau Chan-Vese color (mau Lab)'],
            cols=3, figsize=(15,5))

## 9. Bài tập tổng hợp cuối chương (§9)

### 8.1 Bài 9.1 — Ba hướng phân đoạn trên một ảnh; so sánh ưu nhược điểm

In [ ]:
img = load_image('tablets4.png')
H, W = img.shape[:2]

# Huong 1: Auto-thresholding (Otsu hai buoc S+V)
_, _, mask_thresh = otsu_two_step_sv(img)

# Huong 2: K-means Lab+XY
lbl_km = kmeans_color(img, k=6, use_spatial=True, sw=0.2)
img_km = colorize_labels(lbl_km)

# Huong 3: Edge-to-region
_, _, mask_edge, result_edge, ctrs = edge_to_region(img, lo=60, hi=140,
                                                     close_iter=2, min_area=300)
show_images([img, mask_thresh, img_km, mask_edge],
            ['Goc (vien thuoc)', f'Auto-threshold (Otsu S+V)',
             f'K-means Lab+XY (k=6)', f'Edge-to-region ({len(ctrs)} vung)'],
            cols=4, figsize=(20,5))
print('Uu/nhuoc diem:')
print('  Auto-threshold: nhanh, don gian; chi tach 2 lop, nhy cam voi nhieu')
print('  K-means:        nhieu lop, linh hoat; khong bao dam lien thong khong gian')
print('  Edge-to-region: bien chinh xac; can bien dong kin; nhy cam voi bien gia')

### 8.2 Bài 9.2 — Pipeline HSV + auto-thresholding + biên màu để tách vật thể màu nổi bật

*(Bài toán gốc: tách biển báo màu từ ảnh đường phố. Ở đây dùng ảnh tablets4 — viên thuốc màu trên nền cam — như proxy.)*

**Pipeline:**
```
RGB → HSV → Xác định dải Hue mục tiêu → Mask màu
    → Áp dụng Otsu S để loại vùng xám → Refined mask
    → Color Canny để làm sắc nét biên
    → Morphology cleanup
```

In [ ]:
def pipeline_detect_colored_object(img_rgb, hue_range=None,
                                    s_min=60, v_min=60,
                                    morph_iter=2):
    '''
    Pipeline HSV + auto-thresholding + bien mau (§9.2).
    hue_range: (H_lo, H_hi) trong [0,180] (OpenCV convention).
                None = dung Otsu tren S thay vi loc theo H.
    '''
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H_ch, S_ch, V_ch = cv2.split(hsv)

    # Buoc 1: Loc theo dai mau (neu co) hoac Otsu tren S
    if hue_range is not None:
        h_lo, h_hi = hue_range
        if h_lo <= h_hi:
            mask_h = cv2.inRange(H_ch, h_lo, h_hi)
        else:  # wrap-around (vi du do: [160,180] | [0,10])
            mask_h = cv2.bitwise_or(
                cv2.inRange(H_ch, h_lo, 180),
                cv2.inRange(H_ch, 0,    h_hi))
    else:
        _, mask_h = cv2.threshold(S_ch, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)

    # Buoc 2: Loc them theo S va V toi thieu
    mask_sv = cv2.inRange(hsv, (0, s_min, v_min), (180, 255, 255))
    mask_color = cv2.bitwise_and(mask_h, mask_sv)

    # Buoc 3: Color Canny lam sac net bien
    edge_map, _, _, _, _ = edge_to_region(img_rgb, lo=60, hi=140,
                                           close_iter=1, min_area=100)
    # Morphology: dong kin va lam sach mask mau
    kernel = np.ones((5,5),np.uint8)
    mask_clean = cv2.morphologyEx(mask_color, cv2.MORPH_CLOSE, kernel, iterations=morph_iter)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN,  kernel, iterations=1)

    result = overlay_mask(img_rgb, mask_clean, (0,255,0), 0.4)
    return mask_h, mask_color, mask_clean, result

# Demo 1: Dung Otsu tren S (khong biet truoc mau muc tieu)
img = load_image('tablets4.png')
mh, mc, mk, res = pipeline_detect_colored_object(img, hue_range=None, s_min=50, v_min=50)
show_images([img, mh, mc, mk, res],
            ['Goc', 'Mask S (Otsu)', 'Mask S+V filter', 'Sau cleanup', 'Overlay'],
            cols=3, figsize=(15,10))

# Demo 2: Biet truoc mau do (H~0-10 hoac 160-180 trong OpenCV)
print('Demo 2: Phat hien vat the mau do (huong den vien thuoc mau do/hong)')
mh2, mc2, mk2, res2 = pipeline_detect_colored_object(
    img, hue_range=(160, 10), s_min=60, v_min=60)  # Do: wrap-around
show_images([img, mh2, mc2, mk2, res2],
            ['Goc', 'Mask Hue do', 'Mask+SV', 'Sau cleanup', 'Overlay mau do'],
            cols=3, figsize=(15,10))

# Demo 3: Ap dung cho anh la (phat hien la mau do)
img_l = load_image('leaves.png')
print('Demo 3: Phat hien la mau do trong anh la mua thu')
mh3, mc3, mk3, res3 = pipeline_detect_colored_object(
    img_l, hue_range=(0, 15), s_min=80, v_min=60)
show_images([img_l, mh3, mc3, mk3, res3],
            ['Goc', 'Mask Hue (do/cam)', 'Mask+SV', 'Sau cleanup', 'Overlay'],
            cols=3, figsize=(15,10))

### 8.3 Bài 9.3 — Equalization V vs L\* trước khi phân ngưỡng

In [ ]:
# Vat the mau sang tren nen toi: hat bi (vang/nau tren nen xam)
img_seeds = load_image('seed2.png')

eq_v_img = equalize_hsv_v(img_seeds)
eq_l_img = equalize_lab_l(img_seeds)

_, mask_orig = otsu_threshold_numpy(get_channels(img_seeds)['V'])
_, mask_eq_v = otsu_threshold_numpy(get_channels(eq_v_img)['V'])
_, mask_eq_l = otsu_threshold_numpy(get_channels(eq_l_img)['L*'])

show_images([img_seeds, eq_v_img, eq_l_img, mask_orig, mask_eq_v, mask_eq_l],
            ['Goc', 'Sau Eq V', 'Sau Eq L*',
             'Otsu V (goc)', 'Otsu V (sau Eq V)', 'Otsu L* (sau Eq L*)'],
            cols=3, figsize=(15,10))
print('Equalization truoc khi phan nguong tang tuong phan: Otsu hoat dong tot hon.')

### 8.4 Bài 9.4 — "When grayscale fails but color succeeds" (3 ảnh minh họa)

In [ ]:
print('=== Khi Grayscale that bai, Color thanh cong ===')
print()
cases = [
    (load_image('leaves.png'), 'La mua thu',
     'Bien giua la do/vang/tim bi bo sot boi gray Canny (do sang giong nhau)',
     'Canny S / multi-channel Canny'),
    (load_image('tablets4.png'), 'Vien thuoc',
     'Otsu gray khong phan biet vien thuoc mau va nen cam (do sang tuong tu)',
     'Otsu S + Otsu hai buoc'),
    (load_image('seed2.png'), 'Hat bi',
     'Hat vang-nau va nen xam co do sang gan nhau -> gray Otsu tham vong',
     'Khoang cach Lab tu mau hat mau tieu + Otsu'),
]
for img, name, problem, solution in cases:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask_gray = otsu_threshold_numpy(gray)
    e_gray   = cv2.Canny(cv2.GaussianBlur(gray,(5,5),1.5),50,130)
    _, mask_s = otsu_threshold_numpy(get_channels(img)['S'])
    e_s      = canny_on_channel(img,'S',lo=30,hi=100)
    show_images([img, gray, mask_gray, e_gray, mask_s, e_s],
                [f'{name}','Grayscale',
                 'Otsu Gray (van de)','Canny Gray (van de)',
                 'Otsu S (color-aware)','Canny S (color-aware)'],
                cols=3, figsize=(15,10))
    print(f'[{name}]')
    print(f'  Van de: {problem}')
    print(f'  Giai phap: {solution}')
    print()

### 9.5 Bài tập 9.1 — Phân đoạn trái cây (Colorful Fruit Segmentation)

Sử dụng ảnh `fruit.png` để thực hiện phân đoạn các loại quả khác nhau.

In [ ]:
img_fruit = load_image('fruit.png')
if img_fruit is not None:
    # Approach 1: K-means Lab
    lbl_km = kmeans_color(img_fruit, k=5, use_spatial=True)
    
    # Approach 2: Edge-to-region
    _, _, mask_edge, result_e, _ = edge_to_region(img_fruit, lo=50, hi=120)
    
    show_images([img_fruit, colorize_labels(lbl_km), result_e],
                ['Goc (Trai cay)', 'K-means Lab', 'Edge-based'],
                cols=3, figsize=(18, 6))

### 9.6 Bài tập 9.2 — Trích xuất biển báo giao thông (Street Sign Extraction)

Sử dụng ảnh `street_sign.png` để trích xuất biển báo màu đỏ (STOP) và màu xanh.

In [ ]:
img_sign = load_image('street_sign.png')
if img_sign is not None:
    # Segment red signs
    _, _, mask_red, res_red = pipeline_detect_colored_object(img_sign, hue_range=(160, 10))
    # Segment blue signs (H around 100-130 in OpenCV)
    _, _, mask_blue, res_blue = pipeline_detect_colored_object(img_sign, hue_range=(100, 130))
    
    show_images([img_sign, res_red, res_blue],
                ['Goc (Duong pho)', 'Vien thuoc do/Bien bao do', 'Bien bao xanh'],
                cols=3, figsize=(18, 6))